In [1]:
import pandas as pd
import yfinance as yf

In [3]:
catalyst_features = pd.read_csv("datasets/catalyst_ds.csv")

In [4]:
final_tickers = catalyst_features['company_ticker'].dropna().unique()

In [5]:
# pull company financial features for each ticker

def getFinancialInfo(tickers):
    financial_data = []
    for ticker in tickers:
        company = yf.Ticker(ticker)
        row = {"ticker": ticker}
        for statement in [company.financials, company.balance_sheet, company.cashflow]:
            for col in statement.columns:
                if col.year == 2025:
                    row.update(statement[col])
        financial_data.append(row)
    financial_data = pd.DataFrame(financial_data).reindex(columns = 
            ['ticker', 
            'Total Revenue', 
            'Cost Of Revenue', 
            'Operating Revenue',
            'Cash And Cash Equivalents',
            'Total Debt',
            'Capital Expenditure'])
    return financial_data

In [6]:
df = getFinancialInfo(final_tickers)
df.head()

,ticker,Total Revenue,Cost Of Revenue,Operating Revenue,Cash And Cash Equivalents,Total Debt,Capital Expenditure
0,PRAX,0.0,NaN,0.0,357329000.0,110000.0,-56000.0
1,PSTV,0.0,NaN,0.0,4256000.0,821000.0,-67000.0
2,CUE,27466000.0,NaN,27466000.0,27136000.0,4197000.0,-177000.0
3,CLLS,72949000.0,9797000.0,72949000.0,61533000.0,90733000.0,-3533000.0
4,ALLO,0.0,NaN,0.0,51688000.0,83253000.0,-386000.0


In [7]:
# company drug portfolio size in 2025
catalyst_features_copy=catalyst_features.copy()
catalyst_features_copy['date'] = pd.to_datetime(catalyst_features_copy['date'])
drug_count = catalyst_features_copy[catalyst_features_copy['date'].dt.year == 2025].groupby('company_ticker')['drug_name'].nunique()

# map back to the df company table
df['drug_count'] = df['ticker'].map(drug_count).fillna(0).astype(int)

In [8]:
df.to_csv("datasets/company_ds.csv", index = False)